In [93]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [94]:
np.random.seed(42)

In [95]:
colunas=5
linhas=10

In [96]:
ativos = [a for a in range(colunas)]
retornos = np.random.normal(0, 0.1, size=(linhas,len(ativos)))

In [97]:
retornos.shape

(10, 5)

In [98]:
df = pd.DataFrame(retornos, index=range(linhas), columns=ativos)
df = df.pct_change().dropna().reset_index(drop=True)
df

,0,1,2,3,4
0,-1.471372,-12.421696,0.184882,-1.308250,-3.317114
1,0.979259,-1.294913,-0.684713,3.075367,-4.179220
2,0.213349,1.174719,0.298745,-0.525410,-0.181234
3,-3.606582,-0.777084,-0.785111,0.569064,-0.614543
4,-0.924318,4.097938,4.563572,-0.578425,-0.464175
5,-6.424563,-2.609286,-1.035926,0.760977,-3.819892
6,1.028968,-0.887240,144.190597,0.255717,-0.760668
7,-1.604882,-0.179521,-0.940986,-0.773297,-8.510478
8,-1.974782,-3.688005,-10.140838,-2.141196,0.192434


In [108]:
model = pyo.ConcreteModel()

model.ativos = pyo.Set(initialize=ativos)
model.periodos = pyo.Set(initialize=range(linhas-1))
model.retornos = pyo.Param(model.ativos, initialize=df.mean().to_dict())
model.sigma = pyo.Param(model.ativos, model.ativos, initialize=lambda model,a,b: df.cov().iloc[a,b])
model.x = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)

def obj_rule(model):

    retorno = sum(model.retornos[a] * model.x[a] for a in model.ativos) 
    var = sum(model.sigma[a,b] * model.x[a] * model.x[b] for a in model.ativos for b in model.ativos)
    return retorno/pyo.sqrt(var)
model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)
def res_1(model):
    return sum(model.x[a] for a in model.ativos) == 1
model.ress_1 = pyo.Constraint(rule=res_1)

In [109]:
model.pprint()

2 Set Declarations
    ativos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    5 : {0, 1, 2, 3, 4}
    periodos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    9 : {0, 1, 2, 3, 4, 5, 6, 7, 8}

2 Param Declarations
    retornos : Size=5, Index=ativos, Domain=Any, Default=None, Mutable=False
        Key : Value
          0 :  -1.531658186298974
          1 : -1.8427874885779874
          2 :  15.072246999869078
          3 : -0.0739391381772459
          4 :  -2.406098930070476
    sigma : Size=25, Index=ativos*ativos, Domain=Any, Default=None, Mutable=False
        Key    : Value
        (0, 0) :    5.59082656987352
        (0, 1) :  1.7866667786732129
        (0, 2) :   47.75792382993281
        (0, 3) :  0.3913824482716837
        (0, 4) :  0.9079738873227717
        (1, 0) :  1.7866667786732129
        (1, 1) :  20.677329231813584
        

In [110]:
# ------------------- solver
opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
res = opt.solve(model,tee=True)

InvalidExpressionError: Model objective (obj) contains nonlinear terms that cannot be written to LP format

In [102]:
for a in model.ativos:
    print(f'Ativo {a}: {model.x[a].value:.4f}')

Ativo 0: 0.0000
Ativo 1: 0.0000
Ativo 2: 1.0000
Ativo 3: 0.0000
Ativo 4: 0.0000
